In [ ]:
from pathlib import Path
from itertools import islice

root = Path("/kaggle/input")

print('Veri klasörleri')
for pattern in ("*", "*/*", "*/*/*"):
    folders = sorted(
        p for p in root.glob(pattern) if p.is_dir()
    )
    for folder in folders[:40]:
        print(folder)

print('\nİlk 5 video')
examples = list(islice(root.rglob("*.mp4"), 5))

if examples:
    for path in examples:
        print(path)
else:
    print("MP4 bulunamadı.")

print('\nRaw klasörleri')
raw_dirs = [
    p for p in root.rglob("*")
    if p.is_dir() and "raw" in p.name.lower()
]

for path in raw_dirs[:20]:
    print(path)

if not raw_dirs:
    print("Raw isimli klasör bulunamadı.")

In [ ]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

R3D_VIDEO_ROOT = Path(
    "/kaggle/input/datasets/sabbirahmad653/ucf-crime-datasets/"
    "Raw_Videos_Unified/Raw_Videos_Unified"
)

SPLIT_ROOT = Path(
    "/kaggle/input/datasets/minhajuddinmeraj/"
    "anomalydetectiondatasetucf"
)


for cls in ["Normal", "Fighting", "Assault", "Arson"]:
    folder = R3D_VIDEO_ROOT / cls

    if not folder.is_dir():
        raise FileNotFoundError(f"Sınıf klasörü bulunamadı: {folder}")

    count = sum(1 for _ in folder.rglob("*.mp4"))
    print(f"{cls:<10}: {count} MP4")


for filename in ["Anomaly_Train.txt", "Anomaly_Test.txt"]:
    path = SPLIT_ROOT / filename

    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(
            f"Ayrım dosyası bulunamadı veya boş: {path}"
        )


raw_roots = [R3D_VIDEO_ROOT]
split_roots = [SPLIT_ROOT]

print("\nHam video klasörü:", R3D_VIDEO_ROOT)
print("Ayrım dosyalarının klasörü:", SPLIT_ROOT)
print('Klasörler hazır.')

In [ ]:
from pathlib import Path
from collections import Counter
import json
import random
import re
import torch
from torch.utils.data import Sampler


R3D_SETUP_READY = False

SEED = 42
VAL_RATIO = 0.20
NORMAL_PER_EPOCH = 72

R3D_CLASS_NAMES = ["Normal", "Fighting", "Assault", "Arson"]
R3D_CLASS_TO_IDX = {
    name: i for i, name in enumerate(R3D_CLASS_NAMES)
}

INPUT_ROOT = Path("/kaggle/input")


raw_roots = sorted(
    p for p in INPUT_ROOT.rglob("Raw_Videos_Unified")
    if p.is_dir() and (p / "Normal").is_dir()
)

split_roots = sorted(
    p for p in INPUT_ROOT.rglob("anomalydetectiondatasetucf")
    if (p / "Anomaly_Train.txt").is_file()
    and (p / "Anomaly_Test.txt").is_file()
)

if len(raw_roots) != 1 or len(split_roots) != 1:
    raise RuntimeError(
        "İki Kaggle veri setinin de ekli olduğunu kontrol et.\n"
        f"Ham video klasörleri: {raw_roots}\n"
        f"Ayrım klasörleri: {split_roots}"
    )

R3D_VIDEO_ROOT = raw_roots[0]
SPLIT_ROOT = split_roots[0]


def class_of(name):
    return next(
        (cls for cls in R3D_CLASS_NAMES if name.startswith(cls)),
        None
    )


def read_ids(filename):
    text = (SPLIT_ROOT / filename).read_text(encoding="utf-8-sig")

    
    
    names = set(re.findall(
        r"([^/\\\s]+\.mp4)(?=\s|$)",
        text,
        flags=re.IGNORECASE,
    ))

    selected = {name for name in names if class_of(name) is not None}

    if not selected:
        raise ValueError(f"Ayrım dosyası boş veya okunamadı: {filename}")

    return selected


official_train = read_ids("Anomaly_Train.txt")
official_test = read_ids("Anomaly_Test.txt")

if official_train & official_test:
    raise ValueError("Resmî train ve test listeleri kesişiyor!")


R3D_VIDEO_PATHS = {}

for cls in R3D_CLASS_NAMES:
    for path in sorted((R3D_VIDEO_ROOT / cls).rglob("*.mp4")):
        if path.name in R3D_VIDEO_PATHS:
            raise ValueError(f"Tekrarlanan dosya adı: {path.name}")

        R3D_VIDEO_PATHS[path.name] = path

missing = (official_train | official_test) - set(R3D_VIDEO_PATHS)

if missing:
    raise FileNotFoundError(
        f"{len(missing)} video eksik.\n"
        f"Örnekler: {sorted(missing)[:5]}"
    )


R3D_SPLITS = {
    "train": [],
    "val": [],
    "test": sorted(official_test),
}

for cls in R3D_CLASS_NAMES:
    pool = sorted(
        name for name in official_train if class_of(name) == cls
    )

    if len(pool) < 2:
        raise ValueError(f"{cls} için yeterli eğitim videosu yok.")

    random.Random(f"{SEED}:{cls}").shuffle(pool)

    n_val = max(
        1,
        min(len(pool) - 1, round(len(pool) * VAL_RATIO))
    )

    R3D_SPLITS["val"].extend(pool[:n_val])
    R3D_SPLITS["train"].extend(pool[n_val:])

R3D_SPLITS = {
    split: sorted(names) for split, names in R3D_SPLITS.items()
}

train_set = set(R3D_SPLITS["train"])
val_set = set(R3D_SPLITS["val"])
test_set = set(R3D_SPLITS["test"])

assert not (train_set & val_set)
assert not ((train_set | val_set) & test_set)
assert train_set | val_set == official_train
assert test_set == official_test


R3D_TRAIN_NAMES = list(R3D_SPLITS["train"])
R3D_TRAIN_LABELS = [
    R3D_CLASS_TO_IDX[class_of(name)]
    for name in R3D_TRAIN_NAMES
]


class RotatingNormalSampler(Sampler):
    def __init__(
        self,
        labels,
        normal_label=0,
        normal_per_epoch=72,
        seed=42,
    ):
        self.seed = int(seed)
        self.epoch = 0
        self.normal_per_epoch = int(normal_per_epoch)

        self.normal_order = [
            i for i, label in enumerate(labels)
            if label == normal_label
        ]

        self.event_indices = [
            i for i, label in enumerate(labels)
            if label != normal_label
        ]

        if not 1 <= self.normal_per_epoch <= len(self.normal_order):
            raise ValueError(
                "Normal kotası mevcut eğitim havuzunu aşıyor."
            )

        
        random.Random(f"{self.seed}:normal_pool").shuffle(
            self.normal_order
        )

    def set_epoch(self, epoch):
        if not isinstance(epoch, int) or epoch < 0:
            raise ValueError(
                "Epoch indeksi sıfırdan başlayan tam sayı olmalı."
            )
        self.epoch = epoch

    def __len__(self):
        return self.normal_per_epoch + len(self.event_indices)

    def __iter__(self):
        start = (
            self.epoch * self.normal_per_epoch
        ) % len(self.normal_order)

        selected_normal = [
            self.normal_order[
                (start + i) % len(self.normal_order)
            ]
            for i in range(self.normal_per_epoch)
        ]

        
        selected = selected_normal + self.event_indices

        random.Random(
            f"{self.seed}:epoch:{self.epoch}"
        ).shuffle(selected)

        return iter(selected)


r3d_train_sampler = RotatingNormalSampler(
    labels=R3D_TRAIN_LABELS,
    normal_label=R3D_CLASS_TO_IDX["Normal"],
    normal_per_epoch=NORMAL_PER_EPOCH,
    seed=SEED,
)


r3d_criterion = torch.nn.CrossEntropyLoss()

R3D_SAMPLING_CONFIG = {
    "strategy": "rotating_normal_v1",
    "normal_per_epoch": NORMAL_PER_EPOCH,
    "seed": SEED,
    "class_weights": None,
}


R3D_DATA_DIR = Path("/kaggle/working/ucf_r3d18")
R3D_MANIFEST_PATH = R3D_DATA_DIR / "split_manifest.json"

manifest = {
    "seed": SEED,
    "val_ratio": VAL_RATIO,
    "class_names": R3D_CLASS_NAMES,
    "class_to_idx": R3D_CLASS_TO_IDX,
    "raw_video_root": str(R3D_VIDEO_ROOT),
    "splits": R3D_SPLITS,
}

R3D_DATA_DIR.mkdir(parents=True, exist_ok=True)

if R3D_MANIFEST_PATH.exists():
    saved = json.loads(
        R3D_MANIFEST_PATH.read_text(encoding="utf-8")
    )

    if saved != manifest:
        raise RuntimeError(
            
        )

    print("Mevcut ayrım korundu.")
else:
    with R3D_MANIFEST_PATH.open("x", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

    print("Ayrım kaydedildi.")



indices = list(r3d_train_sampler)

assert len(indices) == len(set(indices))

epoch_counts = Counter(
    R3D_TRAIN_LABELS[i] for i in indices
)

split_counts = {
    split: Counter(class_of(name) for name in names)
    for split, names in R3D_SPLITS.items()
}

print(
    f"\n{'Sınıf':<12}"
    f"{'Train havuzu':>14}"
    f"{'Epoch':>9}"
    f"{'Val':>9}"
    f"{'Test':>9}"
)

for cls in R3D_CLASS_NAMES:
    print(
        f"{cls:<12}"
        f"{split_counts['train'][cls]:>14}"
        f"{epoch_counts[R3D_CLASS_TO_IDX[cls]]:>9}"
        f"{split_counts['val'][cls]:>9}"
        f"{split_counts['test'][cls]:>9}"
    )

print(
    f"{'TOPLAM':<12}"
    f"{len(R3D_TRAIN_NAMES):>14}"
    f"{len(r3d_train_sampler):>9}"
    f"{len(R3D_SPLITS['val']):>9}"
    f"{len(R3D_SPLITS['test']):>9}"
)

normal_count = len(r3d_train_sampler.normal_order)
coverage_epochs = (
    normal_count + NORMAL_PER_EPOCH - 1
) // NORMAL_PER_EPOCH

print(
    f"\nNormal havuzunun tamamı ilk "
    f"{coverage_epochs} epoch'ta dolaşılır."
)

print("Ayrım dosyası:", R3D_MANIFEST_PATH)

R3D_SETUP_READY = True

In [ ]:
import cv2
import math
import json
import statistics
from collections import Counter
from datetime import datetime, timezone
from tqdm.auto import tqdm

R3D_METADATA_READY = False

if not globals().get("R3D_SETUP_READY", False):
    raise RuntimeError("Önce veri ayrımı ve örnekleme hücresini tamamla.")

R3D_VIDEO_META = {}
audit_errors = []


def inspect_video(path):
    cap = None

    try:
        cap = cv2.VideoCapture(str(path))

        if not cap.isOpened():
            raise ValueError("Video açılamadı.")

        fps = float(cap.get(cv2.CAP_PROP_FPS))
        count_raw = float(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if not math.isfinite(fps) or fps <= 0:
            raise ValueError(f"Geçersiz FPS: {fps}")

        if not math.isfinite(count_raw) or count_raw < 1:
            raise ValueError(f"Geçersiz kare sayısı: {count_raw}")

        frame_count = int(round(count_raw))
        duration_s = frame_count / fps

        if not math.isfinite(duration_s) or duration_s <= 0:
            raise ValueError(f"Geçersiz süre: {duration_s}")

        ok, frame = cap.read()

        if not ok or frame is None or frame.size == 0:
            raise ValueError("İlk kare okunamadı.")

        height, width = frame.shape[:2]

        return {
            "fps": fps,
            "frame_count": frame_count,
            "duration_s": duration_s,
            "width": int(width),
            "height": int(height),
        }

    finally:
        if cap is not None:
            cap.release()



audit_items = [
    (split, name)
    for split in ("train", "val")
    for name in R3D_SPLITS[split]
]

for split, name in tqdm(audit_items, desc="Video kontrolü"):
    try:
        path = R3D_VIDEO_PATHS[name]
        info = inspect_video(path)

        R3D_VIDEO_META[name] = {
            **info,
            "path": str(path),
            "split": split,
            "class_name": class_of(name),
        }

    except Exception as exc:
        audit_errors.append({
            "video": name,
            "split": split,
            "error": str(exc),
        })



timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
report_path = R3D_DATA_DIR / f"video_audit_{timestamp}.json"

with report_path.open("x", encoding="utf-8") as f:
    json.dump(
        {
            "check_scope": "metadata_and_first_frame_only",
            "videos": R3D_VIDEO_META,
            "errors": audit_errors,
        },
        f,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    )

print('\nVideo kontrolü')
print("Kontrol edilen:", len(audit_items))
print('Geçerli:', len(R3D_VIDEO_META))
print("Hatalı:", len(audit_errors))
print("Rapor:", report_path)

print(
    f"\n{'Ayrım':<8} {'Sınıf':<12} {'Video':>6} "
    f"{'Toplam dk':>11} {'Medyan sn':>11} {'FPS aralığı':>16}"
)

for split in ("train", "val"):
    for cls in R3D_CLASS_NAMES:
        rows = [
            info for info in R3D_VIDEO_META.values()
            if info["split"] == split and info["class_name"] == cls
        ]

        if not rows:
            continue

        durations = [info["duration_s"] for info in rows]
        fps_values = [info["fps"] for info in rows]

        print(
            f"{split:<8} {cls:<12} {len(rows):>6} "
            f"{sum(durations) / 60:>11.1f} "
            f"{statistics.median(durations):>11.1f} "
            f"{min(fps_values):>7.2f}–{max(fps_values):<7.2f}"
        )

fps_counts = Counter(
    round(info["fps"], 3) for info in R3D_VIDEO_META.values()
)
print("\nEn sık görülen FPS değerleri:", fps_counts.most_common(6))

if audit_errors:
    print("\nİlk hatalar:")
    for error in audit_errors[:10]:
        print(f"- {error['video']}: {error['error']}")

    raise RuntimeError(
        "nok "
        
    )

R3D_METADATA_READY = True
print("\nKontrol tamamlandı")

In [ ]:
import math
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from torchvision.models.video import R3D_18_Weights

R3D_CLIP_READY = False

if not globals().get("R3D_METADATA_READY", False):
    raise RuntimeError("Önce video kontrol hücresini tamamla.")

R3D_CLIP_CONFIG = {
    "num_frames": 16,
    "target_fps": 15.0,
    "image_size": 112,
    "spatial_mode": "letterbox",
    "temporal_mode": "nearest_frame_using_nominal_fps",
    "short_video_padding": "repeat_last_frame",
}


r3d_preset = R3D_18_Weights.KINETICS400_V1.transforms()

R3D_CLIP_CONFIG["mean"] = list(r3d_preset.mean)
R3D_CLIP_CONFIG["std"] = list(r3d_preset.std)

R3D_MEAN = torch.tensor(r3d_preset.mean).view(3, 1, 1, 1)
R3D_STD = torch.tensor(r3d_preset.std).view(3, 1, 1, 1)


def r3d_letterbox_rgb(frame_bgr):
    
    size = R3D_CLIP_CONFIG["image_size"]

    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    height, width = rgb.shape[:2]

    scale = min(size / width, size / height)
    new_w = max(1, round(width * scale))
    new_h = max(1, round(height * scale))

    resized = cv2.resize(
        rgb, (new_w, new_h),
        interpolation=cv2.INTER_LINEAR,
    )

    
    canvas = np.empty((size, size, 3), dtype=np.uint8)
    canvas[:] = np.rint(
        np.array(R3D_CLIP_CONFIG["mean"]) * 255
    ).astype(np.uint8)

    top = (size - new_h) // 2
    left = (size - new_w) // 2
    canvas[top:top + new_h, left:left + new_w] = resized

    return canvas


def r3d_max_start(meta):
    
    sample_span = (
        (R3D_CLIP_CONFIG["num_frames"] - 1)
        / R3D_CLIP_CONFIG["target_fps"]
    )
    last_frame_time = (meta["frame_count"] - 1) / meta["fps"]

    return max(0.0, last_frame_time - sample_span)


def r3d_read_clip(video_name, start_s):
    meta = R3D_VIDEO_META[video_name]
    start_s = float(start_s)
    max_start = r3d_max_start(meta)

    if not math.isfinite(start_s) or not 0 <= start_s <= max_start + 1e-6:
        raise ValueError(
            f"Başlangıç 0–{max_start:.3f} saniye arasında olmalı."
        )

    start_s = min(start_s, max_start)

    
    times = (
        start_s
        + np.arange(R3D_CLIP_CONFIG["num_frames"])
        / R3D_CLIP_CONFIG["target_fps"]
    )

    frame_ids = np.rint(times * meta["fps"]).astype(np.int64)

    
    frame_ids = np.clip(frame_ids, 0, meta["frame_count"] - 1)

    needed = set(frame_ids.tolist())
    first, last = int(frame_ids[0]), int(frame_ids[-1])

    cap = cv2.VideoCapture(meta["path"])
    selected = {}

    try:
        if not cap.isOpened():
            raise RuntimeError(f"Video açılamadı: {video_name}")

        
        if first > 0:
            if not cap.set(cv2.CAP_PROP_POS_FRAMES, first):
                raise RuntimeError(
                    f"Kareye gidilemedi: {video_name}, {first}"
                )

            position = cap.get(cv2.CAP_PROP_POS_FRAMES)

            if not math.isfinite(position) or abs(position - first) > 0.5:
                raise RuntimeError(
                    f"Kare konumu doğrulanamadı: {position}"
                )

        for frame_id in range(first, last + 1):
            ok, frame = cap.read()

            
            if not ok or frame is None or frame.size == 0:
                raise RuntimeError(
                    f"Kare okunamadı: {video_name}, kare {frame_id}"
                )

            if frame_id in needed:
                selected[frame_id] = r3d_letterbox_rgb(frame)

    finally:
        cap.release()

    rgb_frames = np.stack([
        selected[int(i)] for i in frame_ids
    ])

    
    clip = (
        torch.from_numpy(rgb_frames)
        .permute(3, 0, 1, 2)
        .float()
        / 255.0
    )

    clip = ((clip - R3D_MEAN) / R3D_STD).contiguous()

    return clip, rgb_frames, frame_ids



demo_name = next(
    name for name in R3D_TRAIN_NAMES
    if class_of(name) == "Fighting"
)

demo_meta = R3D_VIDEO_META[demo_name]


demo_start = r3d_max_start(demo_meta) / 2

demo_clip, demo_frames, demo_ids = r3d_read_clip(
    demo_name, demo_start
)

assert tuple(demo_clip.shape) == (3, 16, 112, 112)
assert demo_clip.dtype == torch.float32
assert torch.isfinite(demo_clip).all().item()

print("Video:", demo_name)
print('Video etiketi:', class_of(demo_name))
print("Kaynak FPS:", demo_meta["fps"])
print("Hedef FPS:", R3D_CLIP_CONFIG["target_fps"])
print('Başlangıç (sn):', round(demo_start, 3))
print('Kaynak kareler:', demo_ids.tolist())
print("Klip boyutu:", tuple(demo_clip.shape))
print('Batch boyutu:', tuple(demo_clip.unsqueeze(0).shape))

fig, axes = plt.subplots(4, 4, figsize=(10, 10))

for index, ax in enumerate(axes.flat):
    ax.imshow(demo_frames[index])
    source_time = demo_ids[index] / demo_meta["fps"]
    ax.set_title(f"{index + 1}. örnek | {source_time:.2f} sn")
    ax.axis("off")

plt.tight_layout()
plt.show()

R3D_CLIP_READY = True
print('\nKlip hazır.')

In [ ]:
import random
import time
import torch
from collections import Counter
from torch.utils.data import Dataset, DataLoader

R3D_LOADER_READY = False

if not globals().get("R3D_CLIP_READY", False):
    raise RuntimeError("Önce klip okuma hücresini tamamla.")

R3D_LOADER_CONFIG = {
    "clips_per_video": 8,
    "batch_size": 2,
    "num_workers": 0,
    "temporal_sampling": "stratified_jitter",
    "seed": SEED,
}


class R3DTrainVideoDataset(Dataset):
    def __init__(self, names, clips_per_video=8, seed=42):
        self.names = list(names)
        self.clips_per_video = int(clips_per_video)
        self.seed = int(seed)
        self.epoch = 0

        if self.clips_per_video < 1:
            raise ValueError("Video başına klip sayısı en az 1 olmalı.")

        if len(self.names) != len(set(self.names)):
            raise ValueError("Eğitim listesinde tekrarlanan video var.")

        if set(self.names) != set(R3D_SPLITS["train"]):
            raise ValueError(
                "Bu yükleyici yalnızca mevcut train ayrımı içindir."
            )

        if any(name not in R3D_VIDEO_META for name in self.names):
            raise ValueError(
                "Bazı eğitim videolarının kontrol bilgileri eksik."
            )

    def __len__(self):
        return len(self.names)

    def set_epoch(self, epoch):
        if not isinstance(epoch, int) or epoch < 0:
            raise ValueError(
                "Epoch indeksi sıfırdan başlayan tam sayı olmalı."
            )

        self.epoch = epoch

    def clip_starts(self, index):
        name = self.names[index]
        max_start = r3d_max_start(R3D_VIDEO_META[name])

        
        rng = random.Random(
            f"{self.seed}:{self.epoch}:{name}"
        )

        
        
        width = max_start / self.clips_per_video

        return [
            (i + rng.random()) * width
            for i in range(self.clips_per_video)
        ]

    def __getitem__(self, index):
        name = self.names[index]
        starts = self.clip_starts(index)
        clips = []

        for start_s in starts:
            clip, _, _ = r3d_read_clip(name, start_s)
            clips.append(clip)

        return {
            
            "clips": torch.stack(clips),

            
            "label": torch.tensor(
                R3D_CLASS_TO_IDX[class_of(name)],
                dtype=torch.long,
            ),

            "video_name": name,
            "start_times": torch.tensor(
                starts, dtype=torch.float64
            ),
        }



expected_labels = [
    R3D_CLASS_TO_IDX[class_of(name)]
    for name in R3D_TRAIN_NAMES
]

if expected_labels != list(R3D_TRAIN_LABELS):
    raise RuntimeError(
        "Video sırası ile sampler etiketleri uyuşmuyor."
    )

r3d_train_dataset = R3DTrainVideoDataset(
    R3D_TRAIN_NAMES,
    clips_per_video=R3D_LOADER_CONFIG["clips_per_video"],
    seed=R3D_LOADER_CONFIG["seed"],
)

r3d_train_loader = DataLoader(
    r3d_train_dataset,
    batch_size=R3D_LOADER_CONFIG["batch_size"],
    sampler=r3d_train_sampler,
    
    num_workers=R3D_LOADER_CONFIG["num_workers"],
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
    generator=torch.Generator().manual_seed(SEED),
)


def r3d_set_train_epoch(epoch):
    
    
    r3d_train_dataset.set_epoch(epoch)
    r3d_train_sampler.set_epoch(epoch)


r3d_set_train_epoch(0)

epoch_indices = list(r3d_train_sampler)
epoch_counts = Counter(
    class_of(R3D_TRAIN_NAMES[i])
    for i in epoch_indices
)

print('Eğitim yükleyicisi')
print('Eğitim videosu:', len(r3d_train_dataset))
print('Epoch başına video:', len(r3d_train_sampler))
print("Video başına klip:", R3D_LOADER_CONFIG["clips_per_video"])
print("Batch başına video:", R3D_LOADER_CONFIG["batch_size"])
print('Epoch başına batch:', len(r3d_train_loader))

print("\nEpoch 1 sınıf dağılımı:")
for cls in R3D_CLASS_NAMES:
    print(f"{cls:<10}: {epoch_counts[cls]}")

print('\nİlk batch okunuyor.')
started = time.perf_counter()

r3d_probe_batch = next(iter(r3d_train_loader))

elapsed = time.perf_counter() - started

expected_shape = (
    R3D_LOADER_CONFIG["batch_size"],
    R3D_LOADER_CONFIG["clips_per_video"],
    3,
    R3D_CLIP_CONFIG["num_frames"],
    R3D_CLIP_CONFIG["image_size"],
    R3D_CLIP_CONFIG["image_size"],
)

assert tuple(r3d_probe_batch["clips"].shape) == expected_shape
assert r3d_probe_batch["clips"].dtype == torch.float32
assert torch.isfinite(r3d_probe_batch["clips"]).all().item()
assert tuple(r3d_probe_batch["label"].shape) == (
    R3D_LOADER_CONFIG["batch_size"],
)

print("\nKlip tensörü:", tuple(r3d_probe_batch["clips"].shape))
print("Etiket tensörü:", tuple(r3d_probe_batch["label"].shape))
print("Okuma süresi:", round(elapsed, 2), "saniye")

for i, name in enumerate(r3d_probe_batch["video_name"]):
    label_id = int(r3d_probe_batch["label"][i])
    starts = r3d_probe_batch["start_times"][i].tolist()

    print(f"\nVideo: {name}")
    print("Video etiketi:", R3D_CLASS_NAMES[label_id])
    print("Klip başlangıçları (sn):", [
        round(value, 2) for value in starts
    ])

R3D_LOADER_READY = True
print('\nYükleyici hazır.')

In [ ]:
import time
import torch
import torchvision
from torch import nn
from torchvision.models.video import r3d_18, R3D_18_Weights

if not globals().get("R3D_LOADER_READY", False):
    raise RuntimeError("Önce eğitim yükleyicisi kontrolünü tamamla.")

if globals().get("R3D_MODEL_READY", False):
    raise RuntimeError(
        "Model zaten hazır; yeniden başlatılmadı. Sonraki adıma geçebiliriz."
    )

R3D_MODEL_READY = False

if not torch.cuda.is_available():
    raise RuntimeError("CUDA bulunamadı. Kaggle GPU ayarını kontrol et.")

r3d_device = torch.device("cuda:0")

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

R3D_MODEL_CONFIG = {
    "num_classes": len(R3D_CLASS_NAMES),
    "hidden_dim": 128,
    "attention_dim": 64,
    "dropout": 0.3,
    "encoder_chunk_size": 4,
}


class R3DAttentionMIL(nn.Module):
    def __init__(
        self,
        num_classes=4,
        hidden_dim=128,
        attention_dim=64,
        dropout=0.3,
        encoder_chunk_size=4,
        pretrained=True,
    ):
        super().__init__()

        if encoder_chunk_size < 1:
            raise ValueError("encoder_chunk_size en az 1 olmalı.")

        weights = (
            R3D_18_Weights.KINETICS400_V1
            if pretrained else None
        )

        self.backbone = r3d_18(weights=weights)

        feature_dim = self.backbone.fc.in_features

        
        
        self.backbone.fc = nn.Identity()
        self.encoder_chunk_size = int(encoder_chunk_size)

        self.project = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, hidden_dim),
            nn.ReLU(),
        )

        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, attention_dim),
            nn.Tanh(),
            nn.Linear(attention_dim, 1),
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

        self.set_stage("warmup")

    def set_stage(self, stage):
        if stage not in ("warmup", "finetune"):
            raise ValueError("Aşama warmup veya finetune olmalı.")

        self.backbone.requires_grad_(False)

        if stage == "finetune":
            
            self.backbone.layer4.requires_grad_(True)

        self.stage = stage
        self.backbone.eval()

    def train(self, mode=True):
        super().train(mode)

        
        
        self.backbone.eval()

        return self

    def encode_clips(self, clips):
        
        
        features = [
            self.backbone(chunk)
            for chunk in clips.split(
                self.encoder_chunk_size, dim=0
            )
        ]

        return torch.cat(features, dim=0)

    def classify_features(self, features, mask=None):
        
        h = self.project(features)
        scores = self.attention(h).squeeze(-1).float()

        if mask is not None:
            if mask.shape != scores.shape:
                raise ValueError("Maske boyutu [Video, Klip] olmalı.")

            mask = mask.to(
                device=scores.device,
                dtype=torch.bool,
            )

            if not mask.any(dim=1).all().item():
                raise ValueError(
                    "Her videoda en az bir geçerli klip olmalı."
                )

            scores = scores.masked_fill(
                ~mask, float("-inf")
            )

        attention = torch.softmax(scores, dim=1)

        
        pooled = (
            h.float() * attention.unsqueeze(-1)
        ).sum(dim=1)

        logits = self.classifier(pooled)

        return logits, attention

    def forward(self, videos, mask=None):
        
        if videos.ndim != 6 or videos.shape[2] != 3:
            raise ValueError(
                "Girdi [B, K, 3, T, H, W] biçiminde olmalı."
            )

        batch_size, clip_count = videos.shape[:2]

        if batch_size < 1 or clip_count < 1:
            raise ValueError("Video ve klip sayıları en az 1 olmalı.")

        clips = videos.reshape(-1, *videos.shape[2:])

        features = self.encode_clips(clips)
        features = features.reshape(
            batch_size, clip_count, -1
        )

        return self.classify_features(features, mask)


print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(r3d_device))
print('\nR3D-18 yükleniyor.')

r3d_model = R3DAttentionMIL(
    **R3D_MODEL_CONFIG,
    pretrained=True,
).to(r3d_device)

total_params = sum(
    p.numel() for p in r3d_model.parameters()
)
trainable_params = sum(
    p.numel()
    for p in r3d_model.parameters()
    if p.requires_grad
)

print(f"\nToplam parametre: {total_params:,}")
print(f'Warmup sırasında eğitilen parametre: {trainable_params:,}')
print("Sınıf sırası:", R3D_CLASS_NAMES)


r3d_probe_inputs = r3d_probe_batch["clips"].to(
    r3d_device,
    non_blocking=True,
)

r3d_model.eval()

torch.cuda.synchronize(r3d_device)
torch.cuda.reset_peak_memory_stats(r3d_device)
started = time.perf_counter()


with torch.inference_mode():
    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
    ):
        r3d_probe_logits, r3d_probe_attention = r3d_model(
            r3d_probe_inputs
        )

torch.cuda.synchronize(r3d_device)
elapsed = time.perf_counter() - started

batch_size, clip_count = r3d_probe_inputs.shape[:2]

assert tuple(r3d_probe_logits.shape) == (
    batch_size, len(R3D_CLASS_NAMES)
)
assert tuple(r3d_probe_attention.shape) == (
    batch_size, clip_count
)
assert torch.isfinite(r3d_probe_logits).all().item()
assert torch.isfinite(r3d_probe_attention).all().item()
assert torch.allclose(
    r3d_probe_attention.sum(dim=1),
    torch.ones(batch_size, device=r3d_device),
    atol=1e-5,
)

peak_gib = (
    torch.cuda.max_memory_allocated(r3d_device)
    / 1024**3
)

print('\nGPU kontrolü')
print("Girdi:", tuple(r3d_probe_inputs.shape))
print("Video sınıf skorları:", tuple(r3d_probe_logits.shape))
print("Klip dikkat ağırlıkları:", tuple(r3d_probe_attention.shape))
print(
    "Video başına dikkat toplamı:",
    r3d_probe_attention.sum(dim=1).cpu().tolist(),
)
print("İlk ileri geçiş süresi:", round(elapsed, 3), "saniye")
print('Tepe tensor belleği:', round(peak_gib, 3), "GiB")

del r3d_probe_inputs

R3D_MODEL_READY = True
print('\nModel kontrol edildi; eğitim başlamadı.')

In [ ]:
import math
import time
import cv2
import numpy as np
import torch
from contextlib import closing
from tqdm.auto import tqdm

if not globals().get("R3D_MODEL_READY", False):
    raise RuntimeError("Önce GPU model kontrolünü tamamla.")

R3D_VAL_READER_READY = False

R3D_VAL_CONFIG = {
    "stride_frames": 8,
    "clip_batch_size": 16,
    "sampling": "full_video_nominal_fps_with_tail",
}


def r3d_val_starts(meta):
    stride_frames = R3D_VAL_CONFIG["stride_frames"]
    num_frames = R3D_CLIP_CONFIG["num_frames"]

    if not 1 <= stride_frames <= num_frames:
        raise ValueError("Pencere adımı klip uzunluğunu aşamaz.")

    stride_s = (
        stride_frames / R3D_CLIP_CONFIG["target_fps"]
    )
    max_start = r3d_max_start(meta)

    starts = np.arange(
        0.0, max_start, stride_s,
        dtype=np.float64,
    )

    
    return np.concatenate([
        starts[starts < max_start - 1e-8],
        [max_start],
    ])


def r3d_full_clip_batches(video_name):
    meta = R3D_VIDEO_META[video_name]
    starts = r3d_val_starts(meta)

    offsets = (
        np.arange(R3D_CLIP_CONFIG["num_frames"])
        / R3D_CLIP_CONFIG["target_fps"]
    )

    if R3D_VAL_CONFIG["clip_batch_size"] < 1:
        raise ValueError("Klip batch boyutu en az 1 olmalı.")

    cap = cv2.VideoCapture(meta["path"])
    cache, pending = {}, []
    next_frame = 0

    try:
        if not cap.isOpened():
            raise RuntimeError(f"Video açılamadı: {video_name}")

        for start_s in starts:
            ids = np.rint(
                (start_s + offsets) * meta["fps"]
            ).astype(np.int64)

            ids = np.clip(
                ids, 0, meta["frame_count"] - 1
            )
            first, last = int(ids[0]), int(ids[-1])

            
            cache = {
                i: frame
                for i, frame in cache.items()
                if i >= first
            }

            
            while next_frame <= last:
                ok, frame = cap.read()

                if not ok or frame is None or frame.size == 0:
                    raise RuntimeError(
                        f"Kare okunamadı: {video_name}, "
                        f"kare {next_frame}"
                    )

                if next_frame >= first:
                    cache[next_frame] = r3d_letterbox_rgb(frame)

                next_frame += 1

            rgb = np.stack([
                cache[int(i)] for i in ids
            ])

            clip = (
                torch.from_numpy(rgb)
                .permute(3, 0, 1, 2)
                .float() / 255.0
            )
            clip = (
                (clip - R3D_MEAN) / R3D_STD
            ).contiguous()

            pending.append(clip)

            if len(pending) == R3D_VAL_CONFIG["clip_batch_size"]:
                yield torch.stack(pending)
                pending.clear()

        if next_frame != meta["frame_count"]:
            raise RuntimeError("Video sonuna ulaşılamadı.")

        
        extra_ok, _ = cap.read()

        if extra_ok:
            raise RuntimeError(
                f"Video bildirilen kare sayısından uzun: {video_name}. "
                "Metadata yeniden kontrol edilmeli."
            )

        
        if pending:
            yield torch.stack(pending)

    finally:
        cap.release()


@torch.inference_mode()
def r3d_eval_video(video_name, show_progress=False):
    expected = len(
        r3d_val_starts(R3D_VIDEO_META[video_name])
    )

    previous_mode = r3d_model.training
    r3d_model.eval()

    feature_parts = []
    processed = 0

    try:
        with closing(
            r3d_full_clip_batches(video_name)
        ) as stream:
            with tqdm(
                total=expected,
                desc="Tam video kontrolü",
                unit="klip",
                disable=not show_progress,
            ) as bar:
                for clips in stream:
                    inputs = clips.to(r3d_device)

                    with torch.autocast(
                        device_type=r3d_device.type,
                        dtype=torch.float16,
                        enabled=r3d_device.type == "cuda",
                    ):
                        features = r3d_model.encode_clips(inputs)

                    
                    feature_parts.append(
                        features.float().cpu()
                    )

                    processed += clips.shape[0]
                    bar.update(clips.shape[0])

                    del inputs, features

        if processed != expected:
            raise RuntimeError(
                f"Klip sayısı uyuşmuyor: {processed}/{expected}"
            )

        all_features = (
            torch.cat(feature_parts)
            .unsqueeze(0)
            .to(r3d_device)
        )

        
        with torch.autocast(
            device_type=r3d_device.type,
            dtype=torch.float16,
            enabled=r3d_device.type == "cuda",
        ):
            logits, attention = (
                r3d_model.classify_features(all_features)
            )

        if (
            not torch.isfinite(logits).all().item()
            or not torch.isfinite(attention).all().item()
        ):
            raise RuntimeError("Model çıktısında geçersiz değer var.")

        return {
            "logits": logits.float().cpu(),
            "attention": attention.float().cpu(),
            "num_clips": processed,
        }

    finally:
        
        r3d_model.train(previous_mode)



r3d_val_names = list(R3D_SPLITS["val"])

pilot_name = min(
    r3d_val_names,
    key=lambda name: abs(
        R3D_VIDEO_META[name]["duration_s"] - 60.0
    ),
)

pilot_meta = R3D_VIDEO_META[pilot_name]
pilot_starts = r3d_val_starts(pilot_meta)

total_val_clips = sum(
    len(r3d_val_starts(R3D_VIDEO_META[name]))
    for name in r3d_val_names
)

print('Doğrulama videosu:', len(r3d_val_names))
print('Doğrulama penceresi:', total_val_clips)
print("\nKontrol videosu:", pilot_name)
print("Video süresi:", round(pilot_meta["duration_s"], 2), "sn")
print("Beklenen pencere sayısı:", len(pilot_starts))

started = time.perf_counter()

r3d_val_probe = r3d_eval_video(
    pilot_name,
    show_progress=True,
)

elapsed = time.perf_counter() - started

assert tuple(r3d_val_probe["logits"].shape) == (
    1, len(R3D_CLASS_NAMES)
)
assert torch.allclose(
    r3d_val_probe["attention"].sum(dim=1),
    torch.ones(1),
    atol=1e-5,
)

print('\nVideo kontrolü')
print("İşlenen pencere:", r3d_val_probe["num_clips"])
print("İlk pencere başlangıcı:", round(float(pilot_starts[0]), 3), "sn")
print("Son pencere başlangıcı:", round(float(pilot_starts[-1]), 3), "sn")
print(
    'Son kaynak kare zamanı:',
    round((pilot_meta["frame_count"] - 1) / pilot_meta["fps"], 3),
    "sn",
)
print("Video sınıf skorları:", tuple(r3d_val_probe["logits"].shape))
print("Dikkat ağırlıkları:", tuple(r3d_val_probe["attention"].shape))
print('Okuma ve model süresi:', round(elapsed, 2), "sn")

R3D_VAL_READER_READY = True
print('\nVideo okuyucu kontrol edildi.')

In [ ]:
import time
import torch
from torch import nn

if not globals().get("R3D_VAL_READER_READY", False):
    raise RuntimeError("Önce tam video okuyucusu kontrolünü tamamla.")

R3D_BACKWARD_READY = False

R3D_TRAIN_CONFIG = {
    "epochs": 20,
    "warmup_epochs": 3,
    "validate_every": 5,
    "warmup_head_lr": 3e-4,
    "finetune_head_lr": 1e-4,
    "finetune_backbone_lr": 1e-5,
    "weight_decay": 1e-3,
    "grad_clip": 1.0,
    "amp_init_scale": 1024.0,
}


def r3d_backward_check():
    device = r3d_device
    use_cuda = device.type == "cuda"

    gpu_ids = []
    if use_cuda:
        gpu_ids = [
            device.index
            if device.index is not None
            else torch.cuda.current_device()
        ]

    previous_stage = r3d_model.stage
    previous_mode = r3d_model.training
    results = []

    
    inputs = r3d_probe_batch["clips"].to(device)
    targets = r3d_probe_batch["label"].to(device)

    
    criterion = nn.CrossEntropyLoss()

    
    with torch.random.fork_rng(devices=gpu_ids):
        try:
            for stage in ("warmup", "finetune"):
                r3d_model.zero_grad(set_to_none=True)
                r3d_model.set_stage(stage)
                r3d_model.train()

                trainable = [
                    p for p in r3d_model.parameters()
                    if p.requires_grad
                ]

                
                
                probe_optimizer = torch.optim.AdamW(
                    trainable,
                    lr=R3D_TRAIN_CONFIG["warmup_head_lr"],
                )

                probe_scaler = torch.amp.GradScaler(
                    device.type,
                    enabled=use_cuda,
                    init_scale=R3D_TRAIN_CONFIG["amp_init_scale"],
                )

                if use_cuda:
                    torch.cuda.synchronize(device)
                    torch.cuda.reset_peak_memory_stats(device)

                started = time.perf_counter()

                with torch.enable_grad():
                    with torch.autocast(
                        device_type=device.type,
                        dtype=torch.float16,
                        enabled=use_cuda,
                    ):
                        logits, _ = r3d_model(inputs)
                        loss = criterion(logits, targets)

                    if not torch.isfinite(loss).item():
                        raise RuntimeError(
                            f"{stage}: loss geçersiz."
                        )

                    probe_scaler.scale(loss).backward()
                    probe_scaler.unscale_(probe_optimizer)

                grad_norm = torch.nn.utils.clip_grad_norm_(
                    trainable,
                    max_norm=R3D_TRAIN_CONFIG["grad_clip"],
                    error_if_nonfinite=True,
                )

                head_grad = (
                    r3d_model.classifier[1].weight.grad
                )
                layer4_grad = (
                    r3d_model.backbone.layer4[0]
                    .conv1[0].weight.grad
                )

                if (
                    head_grad is None
                    or head_grad.abs().sum().item() == 0
                ):
                    raise RuntimeError(
                        f"{stage}: sınıflandırıcıya gradyan ulaşmadı."
                    )

                if stage == "warmup" and layer4_grad is not None:
                    raise RuntimeError(
                        "Warmup sırasında backbone donuk kalmalı."
                    )

                if stage == "finetune":
                    if (
                        layer4_grad is None
                        or layer4_grad.abs().sum().item() == 0
                    ):
                        raise RuntimeError(
                            "Finetune sırasında layer4'e gradyan ulaşmadı."
                        )

                if any(
                    p.grad is not None
                    for p in r3d_model.parameters()
                    if not p.requires_grad
                ):
                    raise RuntimeError(
                        "Donuk parametrelerde beklenmeyen gradyan var."
                    )

                if use_cuda:
                    torch.cuda.synchronize(device)

                results.append({
                    "stage": stage,
                    "trainable_params": sum(
                        p.numel() for p in trainable
                    ),
                    "loss": float(loss.item()),
                    "grad_norm": float(grad_norm.item()),
                    "layer4_gradient": layer4_grad is not None,
                    "seconds": time.perf_counter() - started,
                    "peak_gib": (
                        torch.cuda.max_memory_allocated(device)
                        / 1024**3
                        if use_cuda else None
                    ),
                })

                
                del logits, loss, head_grad, layer4_grad
                r3d_model.zero_grad(set_to_none=True)
                del probe_optimizer, probe_scaler, trainable

        finally:
            r3d_model.zero_grad(set_to_none=True)
            r3d_model.set_stage(previous_stage)
            r3d_model.train(previous_mode)

    return results


print('Warmup ve finetune gradyan kontrolü.')

r3d_backward_report = r3d_backward_check()

for result in r3d_backward_report:
    print(f"\nAşama: {result['stage']}")
    print(
        "Eğitime açık parametre:",
        f"{result['trainable_params']:,}",
    )
    print("Kontrol loss:", round(result["loss"], 4))
    print('Sınıflandırıcı gradyanı: var')
    print(
        "R3D son blok:",
        'gradyan var' if result['layer4_gradient'] else 'donuk',
    )
    print("Süre:", round(result["seconds"], 3), "sn")

    if result["peak_gib"] is not None:
        print(
            "Tepe tensor belleği:",
            round(result["peak_gib"], 3),
            "GiB",
        )

R3D_BACKWARD_READY = True

print('\nGradyan kontrolü tamamlandı.')
print("Model ağırlıkları değiştirilmedi.")
print('Model aşaması:', r3d_model.stage)

In [ ]:
import json
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import torch
import torchvision
from torch import nn
from tqdm.auto import tqdm


def r3d_metrics(cm, loss_sum):
    cm = cm.to(torch.float64)
    support = cm.sum(1)
    predicted = cm.sum(0)
    tp = cm.diag()
    count = int(cm.sum().item())

    if count == 0:
        raise RuntimeError("Metrik hesaplanacak video yok.")

    precision = tp / predicted.clamp_min(1)
    recall = tp / support.clamp_min(1)
    f1 = 2 * tp / (support + predicted).clamp_min(1)

    return {
        "loss": float(loss_sum / count),
        "accuracy": float(tp.sum().item() / count),
        "macro_f1": float(f1.mean().item()),
        "num_videos": count,
        "confusion_matrix": cm.long().tolist(),
        "per_class": [
            {
                "class": cls,
                "precision": float(precision[i]),
                "recall": float(recall[i]),
                "f1": float(f1[i]),
                "support": int(support[i]),
            }
            for i, cls in enumerate(R3D_CLASS_NAMES)
        ],
    }


def r3d_save_pt(payload, path):
    
    tmp = path.with_suffix(".pt.tmp")
    with tmp.open("wb") as f:
        torch.save(payload, f)
    tmp.replace(path)


def r3d_save_json(payload, path):
    tmp = path.with_suffix(".json.tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(
            payload, f,
            ensure_ascii=False,
            indent=2,
            allow_nan=False,
        )
    tmp.replace(path)


def r3d_train_epoch(epoch_index):
    
    r3d_set_train_epoch(epoch_index)
    r3d_model.train()

    num_classes = len(R3D_CLASS_NAMES)
    cm = torch.zeros(num_classes, num_classes, dtype=torch.long)
    loss_sum, seen_names = 0.0, []

    expected_names = [
        R3D_TRAIN_NAMES[i] for i in r3d_train_sampler
    ]
    trainable = [
        p for p in r3d_model.parameters() if p.requires_grad
    ]

    with tqdm(
        r3d_train_loader,
        desc=f"Train {epoch_index + 1:02d}",
    ) as bar:
        for batch in bar:
            inputs = batch["clips"].to(
                r3d_device, non_blocking=True
            )
            targets = batch["label"].to(
                r3d_device, non_blocking=True
            )

            r3d_optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type=r3d_device.type,
                dtype=torch.float16,
                enabled=r3d_device.type == "cuda",
            ):
                logits, _ = r3d_model(inputs)
                loss = r3d_loss_fn(logits, targets)

            if not torch.isfinite(loss).item():
                raise RuntimeError(
                    f"Geçersiz loss: {batch['video_name']}"
                )

            r3d_scaler.scale(loss).backward()
            r3d_scaler.unscale_(r3d_optimizer)

            torch.nn.utils.clip_grad_norm_(
                trainable,
                R3D_TRAIN_CONFIG["grad_clip"],
                error_if_nonfinite=True,
            )

            
            r3d_scaler.step(r3d_optimizer)
            r3d_scaler.update()

            labels_cpu = targets.detach().cpu()
            predictions = logits.detach().float().argmax(1).cpu()

            encoded = labels_cpu * num_classes + predictions
            cm += torch.bincount(
                encoded, minlength=num_classes**2
            ).reshape_as(cm)

            loss_sum += loss.item() * targets.numel()
            seen_names.extend(batch["video_name"])

            bar.set_postfix(
                loss=f"{loss_sum / len(seen_names):.4f}"
            )
            del inputs, targets, logits, loss

    r3d_optimizer.zero_grad(set_to_none=True)

    if sorted(seen_names) != sorted(expected_names):
        raise RuntimeError(
            "İşlenen videolar sampler seçimiyle uyuşmuyor."
        )

    return r3d_metrics(cm, loss_sum)


@torch.inference_mode()
def r3d_validate_all(epoch):
    num_classes = len(R3D_CLASS_NAMES)
    cm = torch.zeros(num_classes, num_classes, dtype=torch.long)
    loss_sum = 0.0

    with tqdm(
        list(R3D_SPLITS["val"]),
        desc=f"Tam val {epoch:02d}",
        unit="video",
    ) as bar:
        for name in bar:
            bar.set_postfix_str(name, refresh=True)

            
            result = r3d_eval_video(name, show_progress=False)
            target = R3D_CLASS_TO_IDX[class_of(name)]
            logits = result["logits"]

            loss_sum += nn.functional.cross_entropy(
                logits,
                torch.tensor([target], dtype=torch.long),
            ).item()

            prediction = int(logits.argmax(1).item())
            cm[target, prediction] += 1

    return r3d_metrics(cm, loss_sum)


def r3d_checkpoint(epoch, validation_pending):
    return {
        "format_version": 1,
        "epoch": epoch,
        "stage": r3d_model.stage,
        "validation_pending": validation_pending,

        
        "model_class": "R3DAttentionMIL",
        "model_config": dict(R3D_MODEL_CONFIG),
        "model_state_dict": r3d_model.state_dict(),
        "optimizer_state_dict": r3d_optimizer.state_dict(),
        "scaler_state_dict": r3d_scaler.state_dict(),

        "class_names": list(R3D_CLASS_NAMES),
        "class_to_idx": dict(R3D_CLASS_TO_IDX),
        "clip_config": dict(R3D_CLIP_CONFIG),
        "loader_config": dict(R3D_LOADER_CONFIG),
        "sampling_config": dict(R3D_SAMPLING_CONFIG),
        "val_config": dict(R3D_VAL_CONFIG),
        "train_config": dict(R3D_TRAIN_CONFIG),

        "splits": R3D_SPLITS,
        "seed": SEED,
        "validation_epochs": r3d_validation_epochs,
        "best_epoch": r3d_best_epoch,
        "best_val_macro_f1": r3d_best_f1,
        "history": r3d_history,

        "python_rng_state": random.getstate(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_states": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available() else []
        ),
        "loader_rng_state": (
            r3d_train_loader.generator.get_state()
        ),
        "torch_version": str(torch.__version__),
        "torchvision_version": str(torchvision.__version__),
    }



if not globals().get("R3D_BACKWARD_READY", False):
    raise RuntimeError("Önce geri geçiş kontrolünü tamamla.")

if globals().get("R3D_TRAINING_STARTED", False):
    raise RuntimeError(
        "Bu oturumda eğitim zaten başlatıldı. "
        "Yeniden başlatılmadı; devam için last.pt kaydını kullanmalıyız."
    )

epochs = int(R3D_TRAIN_CONFIG["epochs"])
warmup_epochs = int(R3D_TRAIN_CONFIG["warmup_epochs"])
validate_every = int(R3D_TRAIN_CONFIG["validate_every"])

if not 1 <= warmup_epochs < epochs or validate_every < 1:
    raise ValueError("Epoch ve doğrulama ayarlarını kontrol et.")

train_names = set(R3D_SPLITS["train"])
val_names = set(R3D_SPLITS["val"])
test_names = set(R3D_SPLITS["test"])

if (
    not val_names
    or train_names & val_names
    or (train_names | val_names) & test_names
):
    raise RuntimeError("Train/val/test ayrımı geçersiz.")

r3d_validation_epochs = sorted(
    {warmup_epochs, epochs}
    | set(range(validate_every, epochs + 1, validate_every))
)

timestamp = datetime.now(timezone.utc).strftime(
    "%Y%m%d_%H%M%S_%f"
)
R3D_RUN_DIR = (
    Path("/kaggle/working/runs") / f"r3d18_mil_{timestamp}"
)
R3D_RUN_DIR.mkdir(parents=True, exist_ok=False)

r3d_model.set_stage("warmup")

head_params = [
    p for name, p in r3d_model.named_parameters()
    if not name.startswith("backbone.")
]
layer4_params = list(r3d_model.backbone.layer4.parameters())


r3d_optimizer = torch.optim.AdamW(
    [
        {
            "params": head_params,
            "lr": R3D_TRAIN_CONFIG["warmup_head_lr"],
            "name": "head",
        },
        {
            "params": layer4_params,
            "lr": 0.0,
            "name": "backbone_layer4",
        },
    ],
    weight_decay=R3D_TRAIN_CONFIG["weight_decay"],
)

r3d_scaler = torch.amp.GradScaler(
    r3d_device.type,
    enabled=r3d_device.type == "cuda",
    init_scale=R3D_TRAIN_CONFIG["amp_init_scale"],
)


r3d_loss_fn = nn.CrossEntropyLoss()
r3d_history = []
r3d_best_f1, r3d_best_epoch = None, None

print("Çıktı klasörü:", R3D_RUN_DIR)
print("Tam doğrulama epoch'ları:", r3d_validation_epochs)
print(
    f"İlk {warmup_epochs} epoch warmup; "
    "sonrasında R3D layer4 de eğitilecek."
)


r3d_save_pt(
    r3d_checkpoint(0, False),
    R3D_RUN_DIR / "last.pt",
)

R3D_TRAINING_STARTED = True
R3D_TRAINING_FINISHED = False



for epoch in range(1, epochs + 1):
    stage = (
        "warmup" if epoch <= warmup_epochs else "finetune"
    )
    r3d_model.set_stage(stage)

    r3d_optimizer.param_groups[0]["lr"] = R3D_TRAIN_CONFIG[
        "warmup_head_lr"
        if stage == "warmup"
        else "finetune_head_lr"
    ]
    r3d_optimizer.param_groups[1]["lr"] = (
        0.0
        if stage == "warmup"
        else R3D_TRAIN_CONFIG["finetune_backbone_lr"]
    )

    print(f"\nEpoch {epoch:02d}/{epochs} | {stage}")

    started = time.perf_counter()
    train_metrics = r3d_train_epoch(epoch - 1)

    row = {
        "epoch": epoch,
        "stage": stage,
        "train": train_metrics,
        "val": None,
        "train_seconds": time.perf_counter() - started,
        "val_seconds": None,
        "learning_rates": [
            g["lr"] for g in r3d_optimizer.param_groups
        ],
    }
    r3d_history.append(row)

    should_validate = epoch in r3d_validation_epochs

    
    r3d_save_pt(
        r3d_checkpoint(epoch, should_validate),
        R3D_RUN_DIR / "last.pt",
    )
    r3d_save_json(
        r3d_history,
        R3D_RUN_DIR / "history.json",
    )

    print(
        f"Train loss: {train_metrics['loss']:.4f} | "
        f"Train macro-F1: {train_metrics['macro_f1']:.3f} | "
        f"{row['train_seconds']:.1f} sn | last.pt kaydedildi"
    )

    if should_validate:
        started = time.perf_counter()
        val_metrics = r3d_validate_all(epoch)

        row["val"] = val_metrics
        row["val_seconds"] = time.perf_counter() - started

        improved = (
            r3d_best_f1 is None
            or val_metrics["macro_f1"] > r3d_best_f1 + 1e-6
        )

        if improved:
            r3d_best_f1 = val_metrics["macro_f1"]
            r3d_best_epoch = epoch

            r3d_save_pt(
                r3d_checkpoint(epoch, False),
                R3D_RUN_DIR / "best.pt",
            )

        r3d_save_pt(
            r3d_checkpoint(epoch, False),
            R3D_RUN_DIR / "last.pt",
        )
        r3d_save_json(
            r3d_history,
            R3D_RUN_DIR / "history.json",
        )

        print(
            f"Val loss: {val_metrics['loss']:.4f} | "
            f"Val macro-F1: {val_metrics['macro_f1']:.3f} | "
            f"{row['val_seconds'] / 60:.1f} dk"
            + (" | best.pt güncellendi" if improved else "")
        )

        for item in val_metrics["per_class"]:
            print(
                f"{item['class']:<10} "
                f"P: {item['precision']:.3f} "
                f"R: {item['recall']:.3f} "
                f"F1: {item['f1']:.3f}"
            )


R3D_TRAINING_FINISHED = True
r3d_model.eval()

print('\nEğitim tamamlandı.')
print("En iyi epoch:", r3d_best_epoch)
print("En iyi doğrulama macro-F1:", round(r3d_best_f1, 4))
print("En iyi model:", R3D_RUN_DIR / "best.pt")
print("Son model:", R3D_RUN_DIR / "last.pt")
print("Geçmiş:", R3D_RUN_DIR / "history.json")
print('Test öncesi best.pt yüklenmeli.')

In [ ]:
from pathlib import Path
import torch

best_path = Path(
    "/kaggle/working/runs/"
    "r3d18_mil_20260910_214925_400162/best.pt"
)

if not best_path.is_file():
    raise FileNotFoundError(f"Model bulunamadı: {best_path}")

r3d_inspect_checkpoint = torch.load(
    best_path,
    map_location="cpu",
    weights_only=True,
)

saved_epoch = r3d_inspect_checkpoint["epoch"]
saved_classes = r3d_inspect_checkpoint["class_names"]

best_record = next(
    row
    for row in r3d_inspect_checkpoint["history"]
    if row["epoch"] == saved_epoch and row["val"] is not None
)

metrics = best_record["val"]

print("Kaydedilen epoch:", saved_epoch)
print("Validation macro-F1:", round(metrics["macro_f1"], 4))
print('\nSatır: gerçek sınıf | Sütun: tahmin\n')

print(f"{'Gerçek / Tahmin':<18}" + "".join(
    f"{name:>12}" for name in saved_classes
))

for name, row in zip(
    saved_classes,
    metrics["confusion_matrix"],
):
    print(f"{name:<18}" + "".join(
        f"{value:>12d}" for value in row
    ))



del r3d_inspect_checkpoint

In [ ]:
from pathlib import Path
import torch
from tqdm.auto import tqdm

best_path = Path(
    "/kaggle/working/runs/"
    "r3d18_mil_20260910_214925_400162/best.pt"
)

checkpoint = torch.load(
    best_path,
    map_location="cpu",
    weights_only=True,
)

if checkpoint["class_names"] != R3D_CLASS_NAMES:
    raise ValueError("Modelin sınıf sırası mevcut notebook ile uyuşmuyor.")

if checkpoint["splits"]["val"] != R3D_SPLITS["val"]:
    raise ValueError("Modelin validation ayrımı mevcut ayrımla uyuşmuyor.")

r3d_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)
r3d_model.set_stage(checkpoint["stage"])
r3d_model.eval()

print("Yüklenen epoch:", checkpoint["epoch"])
del checkpoint

event_names = [
    name
    for name in R3D_SPLITS["val"]
    if class_of(name) != "Normal"
]

r3d_val_event_errors = []

for name in tqdm(event_names, desc="Olay videoları"):
    result = r3d_eval_video(name, show_progress=False)

    predicted_id = result["logits"].argmax(dim=1).item()
    predicted_class = R3D_CLASS_NAMES[predicted_id]
    true_class = class_of(name)

    if predicted_class != true_class:
        r3d_val_event_errors.append({
            "video": name,
            "gercek": true_class,
            "tahmin": predicted_class,
        })

print(
    f"\nİncelenen: {len(event_names)} video"
    f"\nYanlış sınıflandırılan: {len(r3d_val_event_errors)} video\n"
)

for row in r3d_val_event_errors:
    print(
        f"{row['video']:<35} "
        f"Gerçek: {row['gercek']:<10} "
        f"Tahmin: {row['tahmin']}"
    )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

inspect_name = "Fighting005_x264.mp4"

if inspect_name not in R3D_SPLITS["val"]:
    raise ValueError("İncelenecek video validation kümesinde olmalı.")


inspect_result = r3d_eval_video(
    inspect_name,
    show_progress=True,
)

inspect_meta = R3D_VIDEO_META[inspect_name]
inspect_starts = r3d_val_starts(inspect_meta)
inspect_attention = inspect_result["attention"][0].numpy()

if len(inspect_starts) != len(inspect_attention):
    raise ValueError("Klip zamanları ile dikkat ağırlıkları uyuşmuyor.")

prediction_id = inspect_result["logits"].argmax(dim=1).item()

print("\nVideo:", inspect_name)
print("Gerçek video etiketi:", class_of(inspect_name))
print("Model tahmini:", R3D_CLASS_NAMES[prediction_id])
print("İşlenen klip:", len(inspect_starts))


plt.figure(figsize=(13, 3))
plt.plot(inspect_starts, inspect_attention)
plt.xlabel("Klip başlangıcı (saniye)")
plt.ylabel("MIL dikkat ağırlığı")
plt.title("Bu ağırlıklar olay olasılığı veya olay sınırı değildir.")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()




selected_indices = []

for index in np.argsort(-inspect_attention):
    sufficiently_far = all(
        abs(inspect_starts[index] - inspect_starts[other]) >= 2.0
        for other in selected_indices
    )

    if sufficiently_far:
        selected_indices.append(int(index))

    if len(selected_indices) == 3:
        break

fig, axes = plt.subplots(
    len(selected_indices),
    4,
    figsize=(12, 3 * len(selected_indices)),
    squeeze=False,
)

for row, clip_index in enumerate(selected_indices):
    start = float(inspect_starts[clip_index])

    _, rgb_frames, frame_ids = r3d_read_clip(inspect_name, start)

    print(
        f"Klip {row + 1}: başlangıç={start:.2f} sn | "
        f"ağırlık={inspect_attention[clip_index]:.6f}"
    )

    
    preview_indices = np.linspace(
        0, len(rgb_frames) - 1, 4, dtype=int
    )

    for col, frame_index in enumerate(preview_indices):
        ax = axes[row, col]
        ax.imshow(rgb_frames[frame_index])
        ax.set_title(
            f"Klip {row + 1} | "
            f"{frame_ids[frame_index] / inspect_meta['fps']:.2f} sn"
        )
        ax.axis("off")

plt.tight_layout()
plt.show()